# 🎙️ Voice-Pro trên Google Colab

Notebook chạy [Voice-Pro](https://github.com/nooptr/voice-pro) trên Colab với **GPU NVIDIA (CUDA)** — transcribe (WhisperX) nhanh hơn nhiều so với chạy CPU trên máy Mac.

Notebook đã tự xử lý 3 vấn đề khi cài trên Colab:
1. **Lỗi `pkg_resources`** khi build wheel (setuptools ≥81 đã bỏ `pkg_resources`) → ghim `setuptools<80`.
2. **Chọn GPU không cần gõ tay** (Colab không nhập input được) → đặt biến `GPU_CHOICE=G`.
3. **Link public** để mở UI từ trình duyệt (code Linux mặc định không bật `share`) → vá `launch(share=True)`.

---

## ⚠️ BƯỚC 0 — Bật GPU (BẮT BUỘC)

Trên menu: **Runtime → Change runtime type → Hardware accelerator → `T4 GPU` → Save**.

Không làm bước này thì WhisperX vẫn chạy CPU và chậm như cũ.

## 1️⃣ Kiểm tra đã có GPU chưa

Phải thấy bảng thông tin GPU (Tesla T4). Nếu báo lỗi → bạn chưa chọn T4 runtime (làm lại Bước 0).

In [2]:
!nvidia-smi

Mon Jun  1 12:36:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2️⃣ Tải mã nguồn và áp dụng các bản vá

Clone repo rồi đặt các biến môi trường + vá file. Dòng cuối phải in ra `gradio_interface.launch(share=True)`.

In [5]:
%cd /content
!git clone https://github.com/nooptr/voice-pro.git 2>/dev/null || echo "Da clone roi, bo qua."
%cd /content/voice-pro

# (1) Sua loi pkg_resources: ghim setuptools<80 cho buoc build wheel
!echo "setuptools<80" > /content/build-constraints.txt
%env PIP_CONSTRAINT=/content/build-constraints.txt

# (2) Tu chon GPU = NVIDIA/CUDA (Colab khong go input duoc; G = NVIDIA)
%env GPU_CHOICE=G

# (3) Bat link public gradio.live (code Linux mac dinh launch() KHONG co share)
!sed -i 's/gradio_interface\.launch()/gradio_interface.launch(share=True)/' app/abus_app_voice.py
!grep -n "launch(share=True)" app/abus_app_voice.py   # xac nhan da va

/content
Da clone roi, bo qua.
/content/voice-pro
env: PIP_CONSTRAINT=/content/build-constraints.txt
env: GPU_CHOICE=G
124:        gradio_interface.launch(share=True)


## 3️⃣ Cài đặt và chạy

**Lần đầu rất lâu (~10–25 phút)**: cài Miniconda + thư viện + tải model. Cứ để chạy.

Khi server khởi động xong, tìm dòng:

```
Running on public URL: https://xxxxxxxx.gradio.live
```

👉 **Bấm vào link `gradio.live`** để mở giao diện Voice-Pro.

> Cell này sẽ chạy mãi (đang phục vụ web) — đó là bình thường, đừng dừng khi đang dùng.

In [ ]:
!bash start.sh


  ABUS Launcher [Version 3.0]
  contact: abus.aikorea@gmail.com


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  130M  100  130M    0     0   173M      0 --:--:-- --:--:-- --:--:--  173M
Installing Miniconda to /content/voice-pro/installer_files/conda
PREFIX=/content/voice-pro/installer_files/conda
Unpacking payload ...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/voice-pro/installer_files/conda
Waiting for installation to complete...
Verifying Miniconda i

---

## 📌 Lưu ý về Colab

| Vấn đề | Lưu ý |
|---|---|
| **Phiên tạm thời** | Đóng tab / idle ~90 phút hoặc quá ~12h → Colab ngắt, mất hết. Chạy lại từ Cell 1. |
| **GPU free không chắc có** | Giờ cao điểm Colab free có thể không cấp T4. Bản Pro ổn định hơn. |
| **Tốc độ** | Với T4, transcribe nhanh hơn Mac (Rosetta) nhiều lần; dùng được cả model `large-v3`. |
| **Riêng tư** | File bạn upload sẽ nằm trên máy chủ Google. |
| **Muốn nhanh hơn nữa** | Trong UI đổi model lên `medium`/`large-v3` và `compute_type` = `float16` (GPU chạy float16 rất nhanh). |

Nếu Cell 3 báo lỗi mới, copy đoạn log (kéo lên đủ phần "above" để thấy package nào fail) để được hỗ trợ tiếp.